# TFT (Temporal Fusion Transformer) — Google Colab
**데이터 경로**: `/content/drive/MyDrive/tft_data/final_dataset_tft_ready.csv`  
**출력 경로**: `/content/drive/MyDrive/tft_outputs/`

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 패키지 설치

In [ ]:
!pip install pytorch-forecasting lightning -q

## 3. Config

In [ ]:
import os

# ── 경로 ──────────────────────────────────────────────────────────────────────
RAW_DATA_PATH       = "/content/drive/MyDrive/tft_data/final_dataset_tft_ready.csv"
PROCESSED_DATA_PATH = "/content/drive/MyDrive/tft_outputs/final_dataset_preprocessed.csv"
SCALER_DIR          = "/content/drive/MyDrive/tft_outputs/scalers"
CHECKPOINT_DIR      = "/content/drive/MyDrive/tft_outputs/checkpoints"
LOG_DIR             = "/content/drive/MyDrive/tft_outputs/logs"

for _d in [SCALER_DIR, CHECKPOINT_DIR, LOG_DIR,
           os.path.dirname(PROCESSED_DATA_PATH)]:
    os.makedirs(_d, exist_ok=True)

# ── Train/Val Split ────────────────────────────────────────────────────────────
# target_5d(t) = log(end_{t+5} / end_t) 이므로 train 마지막 row의 target이
# validation 구간 가격을 참조하지 않도록 두 날짜를 분리.
TRAIN_END_DATE = "2024-07-23"   # train feature row 마지막 날짜
VAL_START_DATE = "2024-07-30"   # validation prediction 시작 날짜

# ── 컬럼 정의 ──────────────────────────────────────────────────────────────────
TARGET_COL   = "target_5d"
GROUP_COL    = "ticker"
SECTOR_COL   = "sector"
TIME_IDX_COL = "time_idx"

LOG1P_COLS = ["AUM", "domestic_count", "global_count"]

SECTOR_ZSCORE_COLS = [
    "domestic_mean",
    "domestic_std",
    "domestic_count_log",
    "global_mean",
    "global_std",
    "global_count_log",
]

MACRO_SCALE_COLS = ["usd_krw", "vix_close"]

# ── Feature Group 관리 ────────────────────────────────────────────────────────
PRICE_FEATURES     = ["end", "AUM_log"]
SENTIMENT_FEATURES = ["domestic_mean", "domestic_std", "domestic_count_log",
                      "global_mean", "global_std", "global_count_log"]
MACRO_FEATURES     = ["usd_krw", "vix_close"]
ROLLING_FEATURES   = []   # 추후 추가 예정
CALENDAR_KNOWN_REALS          = []   # 추후 추가 예정
CALENDAR_KNOWN_CATEGORICALS   = []   # 추후 추가 예정

FEATURE_VERSION = "price_macro_sentiment"   # ← 실험할 버전 이름으로 변경
FEATURE_MAP = {
    "price":                        PRICE_FEATURES,
    "price_macro":                  PRICE_FEATURES + MACRO_FEATURES,
    "price_sentiment":              PRICE_FEATURES + SENTIMENT_FEATURES,
    "price_macro_sentiment":        PRICE_FEATURES + MACRO_FEATURES + SENTIMENT_FEATURES,
    "price_macro_sentiment_rolling": PRICE_FEATURES + MACRO_FEATURES + SENTIMENT_FEATURES + ROLLING_FEATURES,
}

TIME_VARYING_UNKNOWN_REALS      = FEATURE_MAP[FEATURE_VERSION]
TIME_VARYING_KNOWN_REALS        = CALENDAR_KNOWN_REALS
TIME_VARYING_KNOWN_CATEGORICALS = CALENDAR_KNOWN_CATEGORICALS

# ── 모델 하이퍼파라미터 C ────────────────────────────────────────────────────────
MAX_ENCODER_LENGTH    = 60
MAX_PREDICTION_LENGTH = 1
BATCH_SIZE            = 64
LEARNING_RATE         = 1e-4

HIDDEN_SIZE            = 32
ATTENTION_HEAD_SIZE    = 2
DROPOUT                = 0.3
HIDDEN_CONTINUOUS_SIZE = 16

MAX_EPOCHS = 50
PATIENCE   = 5

print(f"Config loaded. FEATURE_VERSION={FEATURE_VERSION}")
print(f"  TIME_VARYING_UNKNOWN_REALS  : {TIME_VARYING_UNKNOWN_REALS}")
print(f"  TIME_VARYING_KNOWN_REALS    : {TIME_VARYING_KNOWN_REALS}")
print(f"  TIME_VARYING_KNOWN_CATEGORICALS: {TIME_VARYING_KNOWN_CATEGORICALS}")

## 4. 전처리 (Preprocessing)

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


def load_and_sort(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
    return df


def apply_log1p(df: pd.DataFrame) -> pd.DataFrame:
    for col in LOG1P_COLS:
        df[f"{col}_log"] = np.log1p(df[col])
    return df


def fit_sector_zscore(train_df: pd.DataFrame) -> dict:
    stats = {}
    for col in SECTOR_ZSCORE_COLS:
        stats[col] = {}
        for sector, grp in train_df.groupby(SECTOR_COL):
            m = grp[col].mean()
            s = grp[col].std()
            s = s if s > 0 else 1.0
            stats[col][sector] = {"mean": m, "std": s}
    return stats


def apply_sector_zscore(df: pd.DataFrame, stats: dict) -> pd.DataFrame:
    for col, sector_stats in stats.items():
        def _zscore(row, _stats=sector_stats):
            s = _stats.get(row[SECTOR_COL])
            if s is None:
                return row[col]
            return (row[col] - s["mean"]) / s["std"]
        df[col] = df.apply(_zscore, axis=1)
    return df


def fit_macro_scaler(train_df: pd.DataFrame) -> StandardScaler:
    scaler = StandardScaler()
    scaler.fit(train_df[MACRO_SCALE_COLS])
    return scaler


def apply_macro_scaler(df: pd.DataFrame, scaler: StandardScaler) -> pd.DataFrame:
    df[MACRO_SCALE_COLS] = scaler.transform(df[MACRO_SCALE_COLS])
    return df


def save_scalers(stats: dict, macro_scaler: StandardScaler, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "sector_zscore_stats.pkl"), "wb") as f:
        pickle.dump(stats, f)
    with open(os.path.join(out_dir, "macro_scaler.pkl"), "wb") as f:
        pickle.dump(macro_scaler, f)
    print(f"Scalers saved → {out_dir}")


print("Preprocessing functions defined.")

In [ ]:
print("Loading data...")
df_raw = load_and_sort(RAW_DATA_PATH)
print(f"  Shape: {df_raw.shape}")

print("Applying log1p...")
df_raw = apply_log1p(df_raw)

train_mask = df_raw["date"] <= pd.Timestamp(TRAIN_END_DATE)
train_df_fit = df_raw[train_mask]
print(f"  Train rows: {train_mask.sum()}, Val rows: {(~train_mask).sum()}")

print("Fitting sector z-score (train only)...")
sector_stats = fit_sector_zscore(train_df_fit)
df_raw = apply_sector_zscore(df_raw, sector_stats)

print("Fitting macro StandardScaler (train only)...")
macro_scaler = fit_macro_scaler(train_df_fit)
df_raw = apply_macro_scaler(df_raw, macro_scaler)

df_raw = df_raw.drop(columns=LOG1P_COLS)

os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)
df_raw.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Preprocessed data saved → {PROCESSED_DATA_PATH}")

save_scalers(sector_stats, macro_scaler, SCALER_DIR)

print("\n── Sanity Check ──")
print(f"  Columns: {list(df_raw.columns)}")
nan_counts = df_raw.isnull().sum()
print(f"  NaN:\n{nan_counts[nan_counts > 0]}")
val_df_check = df_raw[~train_mask.values]
print(f"  Val date range: {val_df_check['date'].min().date()} ~ {val_df_check['date'].max().date()}")

## 5. Dataset & DataLoader

In [ ]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

STATIC_CATEGORICALS = [GROUP_COL, SECTOR_COL]  # ticker, sector


def load_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, parse_dates=["date"])
    df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)
    for col in STATIC_CATEGORICALS:
        df[col] = df[col].astype(str)
    df = df.sort_values([GROUP_COL, "date"]).reset_index(drop=True)
    return df


def make_datasets(csv_path: str = PROCESSED_DATA_PATH, batch_size: int = BATCH_SIZE):
    df = load_data(csv_path)

    train_df = df[df["date"] <= pd.Timestamp(TRAIN_END_DATE)].copy()
    val_min_idx = df.loc[df["date"] >= pd.Timestamp(VAL_START_DATE), TIME_IDX_COL].min()

    print("\n── Dataset Split Check ──")
    print(f"  Train date range : {train_df['date'].min().date()} ~ {train_df['date'].max().date()}")
    val_df_check = df[df[TIME_IDX_COL] >= val_min_idx]
    print(f"  Val date range   : {val_df_check['date'].min().date()} ~ {val_df_check['date'].max().date()}")
    print(f"  val_min_idx      : {val_min_idx}")

    train_ds = TimeSeriesDataSet(
        train_df,
        time_idx=TIME_IDX_COL,
        target=TARGET_COL,
        group_ids=[GROUP_COL],
        max_encoder_length=MAX_ENCODER_LENGTH,
        max_prediction_length=MAX_PREDICTION_LENGTH,
        static_categoricals=STATIC_CATEGORICALS,
        static_reals=[],
        time_varying_known_categoricals=TIME_VARYING_KNOWN_CATEGORICALS,
        time_varying_known_reals=TIME_VARYING_KNOWN_REALS,
        time_varying_unknown_reals=TIME_VARYING_UNKNOWN_REALS,
        target_normalizer=GroupNormalizer(
            groups=[GROUP_COL],
            transformation=None,
        ),
        scalers={
            "end": GroupNormalizer(groups=[GROUP_COL], transformation=None),
            "AUM_log": GroupNormalizer(groups=[GROUP_COL], transformation=None),
        },
        allow_missing_timesteps=True,
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )

    val_ds = TimeSeriesDataSet.from_dataset(
        train_ds,
        df,
        predict=False,
        stop_randomization=True,
        min_prediction_idx=val_min_idx,
    )

    train_loader = train_ds.to_dataloader(train=True, batch_size=batch_size, num_workers=2)
    val_loader   = val_ds.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=2)

    return train_ds, val_ds, train_loader, val_loader


print("Dataset functions defined.")

In [ ]:
train_ds, val_ds, train_loader, val_loader = make_datasets()

print(f"Train samples : {len(train_ds)}")
print(f"Val   samples : {len(val_ds)}")

x, y = next(iter(train_loader))
print(f"Batch x keys  : {list(x.keys())}")
print(f"Batch y shape : {y[0].shape}")

## 6. 모델 & Trainer 정의

In [ ]:
import lightning as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import MAE, RMSE, QuantileLoss

QUANTILES = [0.1, 0.5, 0.9]


def build_model(train_ds) -> TemporalFusionTransformer:
    model = TemporalFusionTransformer.from_dataset(
        train_ds,
        hidden_size=HIDDEN_SIZE,
        attention_head_size=ATTENTION_HEAD_SIZE,
        dropout=DROPOUT,
        hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
        loss=QuantileLoss(quantiles=QUANTILES),
        logging_metrics=[MAE(), RMSE()],
        learning_rate=LEARNING_RATE,
        optimizer="adam",
        log_interval=10,
        log_val_interval=1,
    )
    return model


def build_trainer(max_epochs: int = MAX_EPOCHS) -> pl.Trainer:
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        mode="min",
        verbose=True,
    )
    checkpoint = ModelCheckpoint(
        dirpath=os.path.join(CHECKPOINT_DIR, FEATURE_VERSION),
        filename=f"tft-{FEATURE_VERSION}-{{epoch:02d}}-{{val_loss:.4f}}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )
    logger = CSVLogger(save_dir=LOG_DIR, name=FEATURE_VERSION)

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        gradient_clip_val=0.1,
        callbacks=[early_stop, checkpoint],
        logger=logger,
        enable_progress_bar=True,
    )
    return trainer


print("Model / Trainer functions defined.")

## 7. 학습 실행

In [ ]:
print("=" * 60)
print("Building model...")
model = build_model(train_ds)
print(f"  Parameters : {sum(p.numel() for p in model.parameters()):,}")

print("\nBuilding trainer...")
trainer = build_trainer(max_epochs=MAX_EPOCHS)

print("\nTraining...")
print("=" * 60)
trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

print("=" * 60)
print("Training complete.")
best_ckpt = trainer.checkpoint_callback.best_model_path
print(f"  Best checkpoint : {best_ckpt}")
print(f"  Best val_loss   : {trainer.checkpoint_callback.best_model_score:.6f}")

## 8. 학습 결과 확인

In [ ]:
import matplotlib.pyplot as plt

metrics_path = os.path.join(trainer.logger.log_dir, "metrics.csv")
if os.path.exists(metrics_path):
    metrics = pd.read_csv(metrics_path)
    print(metrics.tail(10).to_string(index=False))

    val_loss = metrics.dropna(subset=["val_loss"])
    if not val_loss.empty:
        plt.figure(figsize=(8, 4))
        plt.plot(val_loss["epoch"], val_loss["val_loss"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Val Loss")
        plt.title(f"Validation Loss — {FEATURE_VERSION}")
        plt.tight_layout()
        plt.savefig(os.path.join(trainer.logger.log_dir, "val_loss.png"), dpi=120)
        plt.show()
else:
    print(f"metrics.csv not found at {metrics_path}")

## 9. 실험 결과 누적 저장

In [ ]:
SUMMARY_PATH = "/content/drive/MyDrive/tft_outputs/experiment_summary.csv"

# metrics.csv에서 val_loss 최솟값 row 기준으로 MAE/RMSE 추출
best_val_loss = float(trainer.checkpoint_callback.best_model_score)
best_val_MAE  = None
best_val_RMSE = None
if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    _val = metrics_df.dropna(subset=["val_loss"]).copy()
    if not _val.empty:
        best_row = _val.loc[_val["val_loss"].idxmin()]
        best_val_loss = float(best_row["val_loss"])
        best_val_MAE  = float(best_row["val_MAE"])  if "val_MAE"  in best_row and pd.notna(best_row["val_MAE"])  else None
        best_val_RMSE = float(best_row["val_RMSE"]) if "val_RMSE" in best_row and pd.notna(best_row["val_RMSE"]) else None

row = {
    "timestamp":              pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "feature_version":        FEATURE_VERSION,
    "train_end_date":         TRAIN_END_DATE,
    "val_start_date":         VAL_START_DATE,
    "target_col":             TARGET_COL,
    "max_encoder_length":     MAX_ENCODER_LENGTH,
    "max_prediction_length":  MAX_PREDICTION_LENGTH,
    "hidden_size":            HIDDEN_SIZE,
    "hidden_continuous_size": HIDDEN_CONTINUOUS_SIZE,
    "attention_head_size":    ATTENTION_HEAD_SIZE,
    "dropout":                DROPOUT,
    "learning_rate":          LEARNING_RATE,
    "batch_size":             BATCH_SIZE,
    "unknown_reals":          "|".join(TIME_VARYING_UNKNOWN_REALS),
    "known_reals":            "|".join(TIME_VARYING_KNOWN_REALS),
    "known_categoricals":     "|".join(TIME_VARYING_KNOWN_CATEGORICALS),
    "best_val_loss":          best_val_loss,
    "best_val_MAE":           best_val_MAE,
    "best_val_RMSE":          best_val_RMSE,
    "best_checkpoint":        trainer.checkpoint_callback.best_model_path,
    "log_dir":                trainer.logger.log_dir,
}

new_row_df = pd.DataFrame([row])
if os.path.exists(SUMMARY_PATH):
    summary = pd.concat([pd.read_csv(SUMMARY_PATH), new_row_df], ignore_index=True)
else:
    summary = new_row_df

summary.to_csv(SUMMARY_PATH, index=False)
print(f"Experiment summary saved → {SUMMARY_PATH}")
print(new_row_df.T.to_string(header=False))